# COMBIO on public NHANES iron data

This notebook demonstrates the full SHAP-guided selection procedure on NHANES 2017–18. The test partition is reserved first. `COMBIO.fit` uses only the remaining source data, splitting it internally into training and validation. Run the explicit download cell only if you want the seven public CDC files; they are stored locally in ignored `data/nhanes/`. No participant data belong in Git.


## 1. Install

From this repository root:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[dev]'
python -m jupyter lab examples/nhanes_iron.ipynb
```


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from combio import COMBIO

DATA = Path("data/nhanes")
DATA.mkdir(parents=True, exist_ok=True)


## 2. Obtain the seven public CDC files

This cell reuses existing files. If a file is absent, it downloads that public XPT from CDC and checks that the response is an XPT file. The files remain outside version control.


In [ ]:
files = ["DEMO_J", "CBC_J", "BIOPRO_J", "HSCRP_J", "FETIB_J", "FERTIN_J", "BMX_J"]
for name in files:
    path = DATA / f"{name}.xpt"
    if not path.exists():
        url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{name}.xpt"
        with urlopen(url, timeout=90) as response:
            content = response.read()
        if not content.startswith(b"HEADER RECORD"):
            raise ValueError(f"CDC response was not an XPT file: {name}")
        path.write_bytes(content)
    if not path.read_bytes().startswith(b"HEADER RECORD"):
        raise ValueError(f"Invalid XPT file: {path}")
print("Public input files ready:", len(files))


## 3. Assemble the task cohort

The outcome is an analysis-defined iron-overload proxy. Transferrin saturation and ferritin define it and are not candidate predictors. The panel below is the routine-lab candidate list used for this example.


In [ ]:
def load(name):
    return pd.read_sas(DATA / f"{name}.xpt")

demo = load("DEMO_J")[["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH1"]]
cohort = (demo.merge(load("CBC_J"), on="SEQN")
          .merge(load("BIOPRO_J"), on="SEQN")
          .merge(load("HSCRP_J")[["SEQN", "LBXHSCRP"]], on="SEQN")
          .merge(load("FETIB_J")[["SEQN", "LBDPCT"]], on="SEQN")
          .merge(load("FERTIN_J")[["SEQN", "LBXFER"]], on="SEQN")
          .merge(load("BMX_J")[["SEQN", "BMXBMI", "BMXWAIST"]], on="SEQN"))
cohort = cohort.dropna(subset=["LBDPCT", "LBXFER", "RIAGENDR"])
cohort = cohort.loc[cohort.RIDAGEYR >= 18].reset_index(drop=True)
y = ((cohort.LBDPCT >= 45) &
     np.where(cohort.RIAGENDR == 1, cohort.LBXFER > 300, cohort.LBXFER > 200)).astype(int)
labs = ["LBXMCVSI", "LBXRDW", "LBXPLTSI", "LBXWBCSI", "LBXNEPCT",
        "LBXLYPCT", "LBXHGB", "LBXSATSI", "LBXSASSI", "LBXSGTSI",
        "LBXSAPSI", "LBXSTB", "LBXSAL", "LBXSUA", "LBXSCR", "LBXHSCRP"]
X = cohort[labs].astype(float)
print("Cohort rows:", len(X), "outcome events:", int(y.sum()))


## 4. Reserve test data, then discover the formula

`fit` trains the reference model on its inner training subset. Tree SHAP ranks candidates and identifies clear directions on inner validation. The integer formula is selected by validation AUROC; test labels are not passed to `fit`.


In [ ]:
source_idx, test_idx = train_test_split(
    np.arange(len(X)), test_size=0.30, stratify=y, random_state=42)
model = COMBIO().fit(X.iloc[source_idx], y.iloc[source_idx])
print("Top five:", model.top_features_)
print("SHAP trends:", model.shap_directions_)
print("Exponents:", model.all_exponents_)
print("Formula:", model.formula_)
print("Inner validation AUROC:", round(model.validation_auroc_, 3))


## 5. Apply the frozen formula to the reserved test partition

The primary comparison subset also requires components of the predefined FIB-4, APRI, and De Ritis indices. That extra mask makes its sample size directly comparable with those clinical indices. The formula was already fixed before this cell uses test outcomes.


In [ ]:
test_score = model.score(X.iloc[test_idx])
index_components = ["LBXSASSI", "LBXSATSI", "LBXPLTSI"]
needed = sorted(set(model.top_features_) | set(index_components))
common = np.isfinite(X.iloc[test_idx][needed].to_numpy()).all(axis=1)
print("Common test rows:", int(common.sum()))
print("Common test events:", int(y.iloc[test_idx].to_numpy()[common].sum()))
print("Frozen formula AUROC:", round(roc_auc_score(
    y.iloc[test_idx].to_numpy()[common], test_score[common]), 3))


The score is raw, risk-oriented `log C`. It is not a calibrated probability. Preserve input units and validate any probability mapping separately in the intended population.
